# 05 - Catálogo de Dados

Este notebook documenta as tabelas produzidas ao longo do pipeline do MVP, organizado nas camadas Bronze, Silver e Gold.

O catálogo apresenta a finalidade de cada tabela, sua granularidade, seus principais campos e os tipos de dados armazenados, contribuindo para a governança, rastreabilidade e compreensão do fluxo de dados.

As informações documentadas correspondem às tabelas persistidas no schema `workspace.mvp_sprint_3_anac`.

In [0]:
tabelas_mvp = [
    "bronze_vra",
    "bronze_aerodromos",
    "silver_vra",
    "silver_aerodromos",
    "gold_dim_companhia",
    "gold_dim_tempo",
    "gold_dim_aeroporto",
    "gold_dim_rota",
    "gold_fato_voo"
]

for tabela in tabelas_mvp:
    df = spark.table(f"workspace.mvp_sprint_3_anac.{tabela}")

    print("=" * 80)
    print(f"TABELA: {tabela}")
    print(f"Quantidade de registros: {df.count()}")
    print(f"Quantidade de colunas: {len(df.columns)}")
    print("=" * 80)

    df.printSchema()

    print()

TABELA: bronze_vra
Quantidade de registros: 591447
Quantidade de colunas: 23
root
 |-- sigla_icao_empresa_aerea: string (nullable = true)
 |-- empresa_aerea: string (nullable = true)
 |-- numero_voo: string (nullable = true)
 |-- codigo_di: string (nullable = true)
 |-- codigo_tipo_linha: string (nullable = true)
 |-- modelo_equipamento: string (nullable = true)
 |-- numero_de_assentos: string (nullable = true)
 |-- sigla_icao_aeroporto_origem: string (nullable = true)
 |-- descricao_aeroporto_origem: string (nullable = true)
 |-- partida_prevista: string (nullable = true)
 |-- partida_real: string (nullable = true)
 |-- sigla_icao_aeroporto_destino: string (nullable = true)
 |-- descricao_aeroporto_destino: string (nullable = true)
 |-- chegada_prevista: string (nullable = true)
 |-- chegada_real: string (nullable = true)
 |-- situacao_voo: string (nullable = true)
 |-- justificativa: string (nullable = true)
 |-- referencia: string (nullable = true)
 |-- situacao_partida: string (nul

In [0]:
from pyspark.sql import Row

catalogo_tabelas = [
    Row(
        camada="Bronze",
        tabela="bronze_vra",
        descricao="Dados brutos de Voos Regulares Ativos (VRA) da ANAC, preservados próximos ao formato original e acrescidos de metadados técnicos de ingestão.",
        granularidade="Um registro operacional conforme disponibilizado nos arquivos mensais do VRA."
    ),
    Row(
        camada="Bronze",
        tabela="bronze_aerodromos",
        descricao="Cadastro bruto de aeródromos públicos da ANAC, preservado próximo ao formato original e acrescido de metadados técnicos de ingestão.",
        granularidade="Um registro cadastral de aeródromo conforme a fonte da ANAC."
    ),
    Row(
        camada="Silver",
        tabela="silver_vra",
        descricao="Dados do VRA tratados, deduplicados, tipados e enriquecidos com métricas de atraso, cancelamento e atributos temporais.",
        granularidade="Um registro operacional do VRA após remoção de duplicatas exatas."
    ),
    Row(
        camada="Silver",
        tabela="silver_aerodromos",
        descricao="Cadastro de aeródromos tratado e deduplicado, com coordenadas e altitude convertidas para representação numérica quando possível.",
        granularidade="Um registro cadastral de aeródromo após remoção de duplicatas exatas."
    ),
    Row(
        camada="Gold",
        tabela="gold_dim_companhia",
        descricao="Dimensão contendo código ICAO e nome das companhias aéreas presentes no VRA.",
        granularidade="Uma companhia aérea por código ICAO."
    ),
    Row(
        camada="Gold",
        tabela="gold_dim_tempo",
        descricao="Dimensão temporal utilizada para análises por data, mês e dia da semana.",
        granularidade="Uma linha por data."
    ),
    Row(
        camada="Gold",
        tabela="gold_dim_aeroporto",
        descricao="Dimensão consolidada de aeroportos presentes no VRA, enriquecida com informações cadastrais da base de aeródromos quando disponíveis.",
        granularidade="Um aeroporto por código OACI."
    ),
    Row(
        camada="Gold",
        tabela="gold_dim_rota",
        descricao="Dimensão das rotas direcionais identificadas entre aeroportos de origem e destino.",
        granularidade="Uma combinação direcional única de aeroporto de origem e destino."
    ),
    Row(
        camada="Gold",
        tabela="gold_fato_voo",
        descricao="Tabela fato utilizada nas análises do MVP, contendo atributos operacionais, temporais e indicadores derivados.",
        granularidade="Um registro operacional do VRA."
    )
]

df_catalogo_tabelas = spark.createDataFrame(catalogo_tabelas)

display(df_catalogo_tabelas)

camada,tabela,descricao,granularidade
Bronze,bronze_vra,"Dados brutos de Voos Regulares Ativos (VRA) da ANAC, preservados próximos ao formato original e acrescidos de metadados técnicos de ingestão.",Um registro operacional conforme disponibilizado nos arquivos mensais do VRA.
Bronze,bronze_aerodromos,"Cadastro bruto de aeródromos públicos da ANAC, preservado próximo ao formato original e acrescido de metadados técnicos de ingestão.",Um registro cadastral de aeródromo conforme a fonte da ANAC.
Silver,silver_vra,"Dados do VRA tratados, deduplicados, tipados e enriquecidos com métricas de atraso, cancelamento e atributos temporais.",Um registro operacional do VRA após remoção de duplicatas exatas.
Silver,silver_aerodromos,"Cadastro de aeródromos tratado e deduplicado, com coordenadas e altitude convertidas para representação numérica quando possível.",Um registro cadastral de aeródromo após remoção de duplicatas exatas.
Gold,gold_dim_companhia,Dimensão contendo código ICAO e nome das companhias aéreas presentes no VRA.,Uma companhia aérea por código ICAO.
Gold,gold_dim_tempo,"Dimensão temporal utilizada para análises por data, mês e dia da semana.",Uma linha por data.
Gold,gold_dim_aeroporto,"Dimensão consolidada de aeroportos presentes no VRA, enriquecida com informações cadastrais da base de aeródromos quando disponíveis.",Um aeroporto por código OACI.
Gold,gold_dim_rota,Dimensão das rotas direcionais identificadas entre aeroportos de origem e destino.,Uma combinação direcional única de aeroporto de origem e destino.
Gold,gold_fato_voo,"Tabela fato utilizada nas análises do MVP, contendo atributos operacionais, temporais e indicadores derivados.",Um registro operacional do VRA.


In [0]:
catalogo_campos = []

for tabela in tabelas_mvp:
    df = spark.table(f"workspace.mvp_sprint_3_anac.{tabela}")

    for posicao, campo in enumerate(df.schema.fields, start=1):
        catalogo_campos.append(
            Row(
                tabela=tabela,
                ordem=posicao,
                coluna=campo.name,
                tipo_dado=campo.dataType.simpleString(),
                permite_nulo=campo.nullable
            )
        )

df_catalogo_campos = spark.createDataFrame(catalogo_campos)

display(
    df_catalogo_campos
    .orderBy("tabela", "ordem")
)

tabela,ordem,coluna,tipo_dado,permite_nulo
bronze_aerodromos,1,id_do_aerodromo,string,true
bronze_aerodromos,2,ciad,string,true
bronze_aerodromos,3,codigo_oaci,string,true
bronze_aerodromos,4,id_to_tipo_de_uso,string,true
bronze_aerodromos,5,tipo_de_uso,string,true
bronze_aerodromos,6,nome,string,true
bronze_aerodromos,7,municipio,string,true
bronze_aerodromos,8,uf,string,true
bronze_aerodromos,9,pais,string,true
bronze_aerodromos,10,municipio_servido,string,true


In [0]:
catalogo_semantico_gold = [
    # -------------------------------------------------------------------------
    # gold_dim_companhia
    # -------------------------------------------------------------------------
    Row(
        tabela="gold_dim_companhia",
        coluna="sigla_icao_empresa_aerea",
        descricao="Código ICAO da companhia aérea.",
        dominio_regra="Código textual conforme disponibilizado no VRA.",
        origem_transformacao="silver_vra.sigla_icao_empresa_aerea"
    ),
    Row(
        tabela="gold_dim_companhia",
        coluna="empresa_aerea",
        descricao="Nome da companhia aérea.",
        dominio_regra="Texto conforme disponibilizado no VRA.",
        origem_transformacao="silver_vra.empresa_aerea"
    ),

    # -------------------------------------------------------------------------
    # gold_dim_tempo
    # -------------------------------------------------------------------------
    Row(
        tabela="gold_dim_tempo",
        coluna="data",
        descricao="Data de referência da operação.",
        dominio_regra="Uma data única por registro da dimensão temporal.",
        origem_transformacao="Derivada da data de referência do VRA."
    ),
    Row(
        tabela="gold_dim_tempo",
        coluna="ano",
        descricao="Ano da data de referência.",
        dominio_regra="Ano em formato numérico.",
        origem_transformacao="Derivado de data."
    ),
    Row(
        tabela="gold_dim_tempo",
        coluna="mes",
        descricao="Número do mês da data de referência.",
        dominio_regra="Valores de 1 a 12.",
        origem_transformacao="Derivado de data."
    ),
    Row(
        tabela="gold_dim_tempo",
        coluna="dia",
        descricao="Dia do mês da data de referência.",
        dominio_regra="Valores compatíveis com o calendário.",
        origem_transformacao="Derivado de data."
    ),
    Row(
        tabela="gold_dim_tempo",
        coluna="dia_semana_num",
        descricao="Representação numérica do dia da semana.",
        dominio_regra="Numeração gerada pelas funções temporais do Spark.",
        origem_transformacao="Derivado de data."
    ),
    Row(
        tabela="gold_dim_tempo",
        coluna="dia_semana",
        descricao="Nome do dia da semana.",
        dominio_regra="Categoria textual correspondente à data.",
        origem_transformacao="Derivado de data."
    ),

    # -------------------------------------------------------------------------
    # gold_dim_aeroporto
    # -------------------------------------------------------------------------
    Row(
        tabela="gold_dim_aeroporto",
        coluna="codigo_oaci",
        descricao="Código OACI utilizado para identificar o aeroporto.",
        dominio_regra="Código textual de aeroporto presente nas operações do VRA.",
        origem_transformacao="Códigos de origem e destino do silver_vra."
    ),
    Row(
        tabela="gold_dim_aeroporto",
        coluna="descricao_aeroporto",
        descricao="Descrição textual do aeroporto conforme registrada no VRA.",
        dominio_regra="Texto conforme fonte VRA.",
        origem_transformacao="Descrições de aeroporto de origem e destino do silver_vra."
    ),
    Row(
        tabela="gold_dim_aeroporto",
        coluna="nome",
        descricao="Nome cadastral do aeródromo quando disponível na base de aeródromos da ANAC.",
        dominio_regra="Pode ser nulo para aeroportos sem correspondência na base cadastral utilizada.",
        origem_transformacao="silver_aerodromos.nome, por associação ao código OACI."
    ),
    Row(
        tabela="gold_dim_aeroporto",
        coluna="municipio",
        descricao="Município de localização cadastral do aeródromo.",
        dominio_regra="Pode ser nulo quando não há enriquecimento cadastral disponível.",
        origem_transformacao="silver_aerodromos.municipio"
    ),
    Row(
        tabela="gold_dim_aeroporto",
        coluna="uf",
        descricao="Unidade federativa de localização cadastral do aeródromo.",
        dominio_regra="Sigla de UF quando disponível.",
        origem_transformacao="silver_aerodromos.uf"
    ),
    Row(
        tabela="gold_dim_aeroporto",
        coluna="pais",
        descricao="País associado ao cadastro do aeródromo.",
        dominio_regra="Texto quando disponível na base cadastral.",
        origem_transformacao="silver_aerodromos.pais"
    ),
    Row(
        tabela="gold_dim_aeroporto",
        coluna="municipio_servido",
        descricao="Município servido pelo aeródromo, quando informado pela fonte cadastral.",
        dominio_regra="Campo textual e opcional.",
        origem_transformacao="silver_aerodromos.municipio_servido"
    ),
    Row(
        tabela="gold_dim_aeroporto",
        coluna="uf_servido",
        descricao="Unidade federativa do município servido pelo aeródromo.",
        dominio_regra="Sigla de UF quando disponível.",
        origem_transformacao="silver_aerodromos.uf_servido"
    ),
    Row(
        tabela="gold_dim_aeroporto",
        coluna="latitude_double",
        descricao="Latitude do aeródromo em representação decimal.",
        dominio_regra="Valor numérico convertido a partir da coordenada original.",
        origem_transformacao="silver_aerodromos.latitude_double"
    ),
    Row(
        tabela="gold_dim_aeroporto",
        coluna="longitude_double",
        descricao="Longitude do aeródromo em representação decimal.",
        dominio_regra="Valor numérico convertido a partir da coordenada original.",
        origem_transformacao="silver_aerodromos.longitude_double"
    ),
    Row(
        tabela="gold_dim_aeroporto",
        coluna="altitude_double",
        descricao="Altitude do aeródromo em representação numérica.",
        dominio_regra="Valor numérico quando a conversão do campo original foi possível.",
        origem_transformacao="silver_aerodromos.altitude_double"
    ),

    # -------------------------------------------------------------------------
    # gold_dim_rota
    # -------------------------------------------------------------------------
    Row(
        tabela="gold_dim_rota",
        coluna="aeroporto_origem",
        descricao="Código do aeroporto de origem da rota.",
        dominio_regra="Código de aeroporto presente no VRA.",
        origem_transformacao="silver_vra.sigla_icao_aeroporto_origem"
    ),
    Row(
        tabela="gold_dim_rota",
        coluna="aeroporto_destino",
        descricao="Código do aeroporto de destino da rota.",
        dominio_regra="Código de aeroporto presente no VRA.",
        origem_transformacao="silver_vra.sigla_icao_aeroporto_destino"
    ),
    Row(
        tabela="gold_dim_rota",
        coluna="rota_id",
        descricao="Identificador textual da rota direcional.",
        dominio_regra="Formato aeroporto_origem->aeroporto_destino.",
        origem_transformacao="Concatenação dos códigos de origem e destino."
    ),

    # -------------------------------------------------------------------------
    # gold_fato_voo
    # -------------------------------------------------------------------------
    Row(
        tabela="gold_fato_voo",
        coluna="sigla_icao_empresa_aerea",
        descricao="Código ICAO da companhia aérea associada ao registro operacional.",
        dominio_regra="Código textual presente no VRA.",
        origem_transformacao="silver_vra.sigla_icao_empresa_aerea"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="numero_voo",
        descricao="Número do voo registrado no VRA.",
        dominio_regra="Valor textual conforme fonte.",
        origem_transformacao="silver_vra.numero_voo"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="codigo_di",
        descricao="Código DI conforme disponibilizado no VRA.",
        dominio_regra="Valor textual preservado da fonte.",
        origem_transformacao="silver_vra.codigo_di"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="codigo_tipo_linha",
        descricao="Código do tipo de linha conforme disponibilizado no VRA.",
        dominio_regra="Categoria textual conforme fonte.",
        origem_transformacao="silver_vra.codigo_tipo_linha"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="modelo_equipamento",
        descricao="Código ou identificação do modelo de equipamento utilizado na operação.",
        dominio_regra="Valor textual conforme VRA.",
        origem_transformacao="silver_vra.modelo_equipamento"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="numero_de_assentos_int",
        descricao="Número de assentos do equipamento convertido para tipo inteiro.",
        dominio_regra="Valor inteiro quando a conversão do campo original foi possível.",
        origem_transformacao="Conversão de silver_vra.numero_de_assentos."
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="codigo_aeroporto_origem",
        descricao="Código do aeroporto de origem da operação.",
        dominio_regra="Código textual de aeroporto.",
        origem_transformacao="silver_vra.sigla_icao_aeroporto_origem"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="codigo_aeroporto_destino",
        descricao="Código do aeroporto de destino da operação.",
        dominio_regra="Código textual de aeroporto.",
        origem_transformacao="silver_vra.sigla_icao_aeroporto_destino"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="data",
        descricao="Data de referência utilizada para relacionamento com a dimensão temporal.",
        dominio_regra="Tipo date.",
        origem_transformacao="Derivada da referência temporal do silver_vra."
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="partida_prevista_ts",
        descricao="Data e hora previstas para a partida.",
        dominio_regra="Timestamp quando informado e convertido com sucesso.",
        origem_transformacao="silver_vra.partida_prevista_ts"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="partida_real_ts",
        descricao="Data e hora efetivamente registradas para a partida.",
        dominio_regra="Timestamp; pode ser nulo, inclusive em registros cancelados.",
        origem_transformacao="silver_vra.partida_real_ts"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="chegada_prevista_ts",
        descricao="Data e hora previstas para a chegada.",
        dominio_regra="Timestamp quando informado e convertido com sucesso.",
        origem_transformacao="silver_vra.chegada_prevista_ts"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="chegada_real_ts",
        descricao="Data e hora efetivamente registradas para a chegada.",
        dominio_regra="Timestamp; pode ser nulo, inclusive em registros cancelados.",
        origem_transformacao="silver_vra.chegada_real_ts"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="atraso_partida_min",
        descricao="Diferença, em minutos, entre a partida realizada e a partida prevista.",
        dominio_regra="Valores positivos representam atraso e valores negativos representam antecipação.",
        origem_transformacao="Calculado a partir de partida_real_ts e partida_prevista_ts."
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="atraso_chegada_min",
        descricao="Diferença, em minutos, entre a chegada realizada e a chegada prevista.",
        dominio_regra="Valores positivos representam atraso e valores negativos representam antecipação.",
        origem_transformacao="Calculado a partir de chegada_real_ts e chegada_prevista_ts."
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="flag_atraso_30",
        descricao="Indicador de atraso de partida superior a 30 minutos.",
        dominio_regra="1 para atraso superior a 30 minutos; 0 caso contrário; nulo quando o atraso não pode ser calculado.",
        origem_transformacao="Derivado de atraso_partida_min na camada Silver."
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="flag_atraso_60",
        descricao="Indicador de atraso de partida superior a 60 minutos.",
        dominio_regra="1 para atraso superior a 60 minutos; 0 caso contrário; nulo quando o atraso não pode ser calculado.",
        origem_transformacao="Derivado de atraso_partida_min na camada Silver."
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="flag_cancelado",
        descricao="Indicador de cancelamento do registro operacional.",
        dominio_regra="1 quando situacao_voo é CANCELADO; 0 caso contrário.",
        origem_transformacao="Derivado de situacao_voo na camada Silver."
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="flag_atraso_extremo",
        descricao="Indicador criado para identificar registros com desvios temporais considerados extremos durante a análise de qualidade.",
        dominio_regra="Indicador binário ou nulo quando não existem valores temporais suficientes para avaliação.",
        origem_transformacao="Derivado dos atrasos de partida e chegada na camada Silver."
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="situacao_voo",
        descricao="Situação geral do voo conforme registrada pela ANAC.",
        dominio_regra="Categorias observadas incluem REALIZADO e CANCELADO.",
        origem_transformacao="silver_vra.situacao_voo"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="situacao_partida",
        descricao="Classificação da situação da partida conforme a fonte.",
        dominio_regra="Categorias como Pontual, Antecipado e faixas de atraso.",
        origem_transformacao="silver_vra.situacao_partida"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="situacao_chegada",
        descricao="Classificação da situação da chegada conforme a fonte.",
        dominio_regra="Categorias como Pontual, Antecipado e faixas de atraso.",
        origem_transformacao="silver_vra.situacao_chegada"
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="hora_partida_prevista",
        descricao="Hora do dia extraída do horário previsto de partida.",
        dominio_regra="Valor inteiro entre 0 e 23 quando o horário previsto está disponível.",
        origem_transformacao="Derivada de partida_prevista_ts."
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="periodo_dia",
        descricao="Faixa do dia associada ao horário previsto de partida.",
        dominio_regra="Madrugada, Manhã, Tarde, Noite ou Não informado.",
        origem_transformacao="Derivada de hora_partida_prevista na camada Silver."
    ),
    Row(
        tabela="gold_fato_voo",
        coluna="rota_id",
        descricao="Identificador da rota direcional da operação.",
        dominio_regra="Formato codigo_aeroporto_origem->codigo_aeroporto_destino.",
        origem_transformacao="Concatenação dos códigos dos aeroportos de origem e destino."
    )
]

df_catalogo_semantico_gold = spark.createDataFrame(catalogo_semantico_gold)

catalogo_gold_completo = (
    df_catalogo_semantico_gold
    .join(
        df_catalogo_campos.select(
            "tabela",
            "ordem",
            "coluna",
            "tipo_dado",
            "permite_nulo"
        ),
        on=["tabela", "coluna"],
        how="left"
    )
    .select(
        "tabela",
        "ordem",
        "coluna",
        "tipo_dado",
        "permite_nulo",
        "descricao",
        "dominio_regra",
        "origem_transformacao"
    )
    .orderBy("tabela", "ordem")
)

display(catalogo_gold_completo)

tabela,ordem,coluna,tipo_dado,permite_nulo,descricao,dominio_regra,origem_transformacao
gold_dim_aeroporto,1,codigo_oaci,string,true,Código OACI utilizado para identificar o aeroporto.,Código textual de aeroporto presente nas operações do VRA.,Códigos de origem e destino do silver_vra.
gold_dim_aeroporto,2,descricao_aeroporto,string,true,Descrição textual do aeroporto conforme registrada no VRA.,Texto conforme fonte VRA.,Descrições de aeroporto de origem e destino do silver_vra.
gold_dim_aeroporto,3,nome,string,true,Nome cadastral do aeródromo quando disponível na base de aeródromos da ANAC.,Pode ser nulo para aeroportos sem correspondência na base cadastral utilizada.,"silver_aerodromos.nome, por associação ao código OACI."
gold_dim_aeroporto,4,municipio,string,true,Município de localização cadastral do aeródromo.,Pode ser nulo quando não há enriquecimento cadastral disponível.,silver_aerodromos.municipio
gold_dim_aeroporto,5,uf,string,true,Unidade federativa de localização cadastral do aeródromo.,Sigla de UF quando disponível.,silver_aerodromos.uf
gold_dim_aeroporto,6,pais,string,true,País associado ao cadastro do aeródromo.,Texto quando disponível na base cadastral.,silver_aerodromos.pais
gold_dim_aeroporto,7,municipio_servido,string,true,"Município servido pelo aeródromo, quando informado pela fonte cadastral.",Campo textual e opcional.,silver_aerodromos.municipio_servido
gold_dim_aeroporto,8,uf_servido,string,true,Unidade federativa do município servido pelo aeródromo.,Sigla de UF quando disponível.,silver_aerodromos.uf_servido
gold_dim_aeroporto,9,latitude_double,double,true,Latitude do aeródromo em representação decimal.,Valor numérico convertido a partir da coordenada original.,silver_aerodromos.latitude_double
gold_dim_aeroporto,10,longitude_double,double,true,Longitude do aeródromo em representação decimal.,Valor numérico convertido a partir da coordenada original.,silver_aerodromos.longitude_double


In [0]:
comentarios_tabelas_gold = {
    "gold_dim_companhia": "Dimensão de companhias aéreas presentes nos registros VRA utilizados pelo MVP.",
    "gold_dim_tempo": "Dimensão temporal utilizada nas análises por data, mês e dia da semana.",
    "gold_dim_aeroporto": "Dimensão consolidada de aeroportos presentes no VRA, enriquecida com dados cadastrais de aeródromos quando disponíveis.",
    "gold_dim_rota": "Dimensão de rotas direcionais formadas pelas combinações de aeroportos de origem e destino.",
    "gold_fato_voo": "Tabela fato do modelo dimensional, com granularidade correspondente a um registro operacional do VRA."
}

for tabela, comentario in comentarios_tabelas_gold.items():
    comentario_sql = comentario.replace("'", "''")

    spark.sql(
        f"""
        COMMENT ON TABLE workspace.mvp_sprint_3_anac.{tabela}
        IS '{comentario_sql}'
        """
    )

for registro in catalogo_semantico_gold:
    tabela = registro["tabela"]
    coluna = registro["coluna"]
    descricao = registro["descricao"].replace("'", "''")

    spark.sql(
        f"""
        ALTER TABLE workspace.mvp_sprint_3_anac.{tabela}
        ALTER COLUMN `{coluna}`
        COMMENT '{descricao}'
        """
    )

print("Comentários das tabelas e colunas Gold registrados com sucesso no Unity Catalog.")

Comentários das tabelas e colunas Gold registrados com sucesso no Unity Catalog.


In [0]:
for tabela in [
    "gold_dim_companhia",
    "gold_dim_tempo",
    "gold_dim_aeroporto",
    "gold_dim_rota",
    "gold_fato_voo"
]:
    print("=" * 80)
    print(f"TABELA: {tabela}")
    print("=" * 80)

    display(
        spark.sql(
            f"DESCRIBE TABLE EXTENDED workspace.mvp_sprint_3_anac.{tabela}"
        )
    )

TABELA: gold_dim_companhia


col_name,data_type,comment
sigla_icao_empresa_aerea,string,Código ICAO da companhia aérea.
empresa_aerea,string,Nome da companhia aérea.
,,
# Delta Statistics Columns,,
Column Names,"sigla_icao_empresa_aerea, empresa_aerea",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspace,
Database,mvp_sprint_3_anac,


TABELA: gold_dim_tempo


col_name,data_type,comment
data,date,Data de referência da operação.
ano,int,Ano da data de referência.
mes,int,Número do mês da data de referência.
dia,int,Dia do mês da data de referência.
dia_semana_num,int,Representação numérica do dia da semana.
dia_semana,string,Nome do dia da semana.
,,
# Delta Statistics Columns,,
Column Names,"mes, data, dia_semana_num, ano, dia_semana, dia",
Column Selection Method,first-32,


TABELA: gold_dim_aeroporto


col_name,data_type,comment
codigo_oaci,string,Código OACI utilizado para identificar o aeroporto.
descricao_aeroporto,string,Descrição textual do aeroporto conforme registrada no VRA.
nome,string,Nome cadastral do aeródromo quando disponível na base de aeródromos da ANAC.
municipio,string,Município de localização cadastral do aeródromo.
uf,string,Unidade federativa de localização cadastral do aeródromo.
pais,string,País associado ao cadastro do aeródromo.
municipio_servido,string,"Município servido pelo aeródromo, quando informado pela fonte cadastral."
uf_servido,string,Unidade federativa do município servido pelo aeródromo.
latitude_double,double,Latitude do aeródromo em representação decimal.
longitude_double,double,Longitude do aeródromo em representação decimal.


TABELA: gold_dim_rota


col_name,data_type,comment
aeroporto_origem,string,Código do aeroporto de origem da rota.
aeroporto_destino,string,Código do aeroporto de destino da rota.
rota_id,string,Identificador textual da rota direcional.
,,
# Delta Statistics Columns,,
Column Names,"aeroporto_origem, aeroporto_destino, rota_id",
Column Selection Method,first-32,
,,
# Detailed Table Information,,
Catalog,workspace,


TABELA: gold_fato_voo


col_name,data_type,comment
sigla_icao_empresa_aerea,string,Código ICAO da companhia aérea associada ao registro operacional.
numero_voo,string,Número do voo registrado no VRA.
codigo_di,string,Código DI conforme disponibilizado no VRA.
codigo_tipo_linha,string,Código do tipo de linha conforme disponibilizado no VRA.
modelo_equipamento,string,Código ou identificação do modelo de equipamento utilizado na operação.
numero_de_assentos_int,int,Número de assentos do equipamento convertido para tipo inteiro.
codigo_aeroporto_origem,string,Código do aeroporto de origem da operação.
codigo_aeroporto_destino,string,Código do aeroporto de destino da operação.
data,date,Data de referência utilizada para relacionamento com a dimensão temporal.
partida_prevista_ts,timestamp,Data e hora previstas para a partida.


In [0]:
catalogo_semantico_silver = [
    # silver_vra - campos derivados/transformados
    Row(
        tabela="silver_vra",
        coluna="partida_prevista_ts",
        descricao="Data e hora previstas para a partida convertidas para timestamp.",
        dominio_regra="Timestamp ou nulo quando o valor original não está disponível.",
        origem_transformacao="Conversão de bronze_vra.partida_prevista."
    ),
    Row(
        tabela="silver_vra",
        coluna="partida_real_ts",
        descricao="Data e hora efetivamente registradas para a partida convertidas para timestamp.",
        dominio_regra="Timestamp ou nulo.",
        origem_transformacao="Conversão de bronze_vra.partida_real."
    ),
    Row(
        tabela="silver_vra",
        coluna="chegada_prevista_ts",
        descricao="Data e hora previstas para a chegada convertidas para timestamp.",
        dominio_regra="Timestamp ou nulo.",
        origem_transformacao="Conversão de bronze_vra.chegada_prevista."
    ),
    Row(
        tabela="silver_vra",
        coluna="chegada_real_ts",
        descricao="Data e hora efetivamente registradas para a chegada convertidas para timestamp.",
        dominio_regra="Timestamp ou nulo.",
        origem_transformacao="Conversão de bronze_vra.chegada_real."
    ),
    Row(
        tabela="silver_vra",
        coluna="referencia_ts",
        descricao="Data de referência do registro convertida para timestamp.",
        dominio_regra="Timestamp derivado do campo referencia.",
        origem_transformacao="Conversão de bronze_vra.referencia."
    ),
    Row(
        tabela="silver_vra",
        coluna="numero_de_assentos_int",
        descricao="Número de assentos convertido para tipo inteiro.",
        dominio_regra="Inteiro ou nulo quando a conversão não é possível.",
        origem_transformacao="Conversão de bronze_vra.numero_de_assentos."
    ),
    Row(
        tabela="silver_vra",
        coluna="atraso_partida_min",
        descricao="Diferença em minutos entre partida real e partida prevista.",
        dominio_regra="Positivo para atraso, negativo para antecipação e nulo quando não calculável.",
        origem_transformacao="Diferença entre partida_real_ts e partida_prevista_ts."
    ),
    Row(
        tabela="silver_vra",
        coluna="atraso_chegada_min",
        descricao="Diferença em minutos entre chegada real e chegada prevista.",
        dominio_regra="Positivo para atraso, negativo para antecipação e nulo quando não calculável.",
        origem_transformacao="Diferença entre chegada_real_ts e chegada_prevista_ts."
    ),
    Row(
        tabela="silver_vra",
        coluna="flag_atraso_30",
        descricao="Indicador de atraso de partida superior a 30 minutos.",
        dominio_regra="1 para atraso > 30 minutos; 0 caso contrário; nulo quando não calculável.",
        origem_transformacao="Derivado de atraso_partida_min."
    ),
    Row(
        tabela="silver_vra",
        coluna="flag_atraso_60",
        descricao="Indicador de atraso de partida superior a 60 minutos.",
        dominio_regra="1 para atraso > 60 minutos; 0 caso contrário; nulo quando não calculável.",
        origem_transformacao="Derivado de atraso_partida_min."
    ),
    Row(
        tabela="silver_vra",
        coluna="flag_cancelado",
        descricao="Indicador de cancelamento da operação.",
        dominio_regra="1 quando situacao_voo = CANCELADO; 0 caso contrário.",
        origem_transformacao="Derivado de situacao_voo."
    ),
    Row(
        tabela="silver_vra",
        coluna="flag_atraso_extremo",
        descricao="Indicador de desvio temporal extremo utilizado na etapa de qualidade.",
        dominio_regra="1 quando atraso ou antecipação ultrapassa os limites definidos; 0 caso contrário; nulo quando não avaliável.",
        origem_transformacao="Derivado de atraso_partida_min e atraso_chegada_min."
    ),
    Row(
        tabela="silver_vra",
        coluna="ano",
        descricao="Ano derivado da data de referência.",
        dominio_regra="Valor inteiro.",
        origem_transformacao="Derivado de referencia_ts."
    ),
    Row(
        tabela="silver_vra",
        coluna="mes",
        descricao="Mês derivado da data de referência.",
        dominio_regra="Valores de 1 a 12.",
        origem_transformacao="Derivado de referencia_ts."
    ),
    Row(
        tabela="silver_vra",
        coluna="dia_semana_num",
        descricao="Número do dia da semana.",
        dominio_regra="Numeração retornada pelas funções temporais do Spark.",
        origem_transformacao="Derivado de referencia_ts."
    ),
    Row(
        tabela="silver_vra",
        coluna="hora_partida_prevista",
        descricao="Hora extraída do horário previsto de partida.",
        dominio_regra="Inteiro de 0 a 23 ou nulo.",
        origem_transformacao="Derivado de partida_prevista_ts."
    ),
    Row(
        tabela="silver_vra",
        coluna="periodo_dia",
        descricao="Faixa do dia correspondente ao horário previsto de partida.",
        dominio_regra="Madrugada, Manhã, Tarde, Noite ou Não informado.",
        origem_transformacao="Derivado de hora_partida_prevista."
    ),

    # silver_aerodromos
    Row(
        tabela="silver_aerodromos",
        coluna="latitude_double",
        descricao="Latitude convertida para representação decimal.",
        dominio_regra="Valor double ou nulo quando a conversão não é possível.",
        origem_transformacao="Conversão de bronze_aerodromos.latitude."
    ),
    Row(
        tabela="silver_aerodromos",
        coluna="longitude_double",
        descricao="Longitude convertida para representação decimal.",
        dominio_regra="Valor double ou nulo quando a conversão não é possível.",
        origem_transformacao="Conversão de bronze_aerodromos.longitude."
    ),
    Row(
        tabela="silver_aerodromos",
        coluna="altitude_double",
        descricao="Altitude convertida para representação numérica.",
        dominio_regra="Valor double ou nulo quando a conversão não é possível.",
        origem_transformacao="Conversão de bronze_aerodromos.altitude."
    )
]

df_catalogo_semantico_silver = spark.createDataFrame(catalogo_semantico_silver)

display(
    df_catalogo_semantico_silver
    .join(
        df_catalogo_campos.select(
            "tabela", "ordem", "coluna", "tipo_dado", "permite_nulo"
        ),
        on=["tabela", "coluna"],
        how="left"
    )
    .select(
        "tabela",
        "ordem",
        "coluna",
        "tipo_dado",
        "permite_nulo",
        "descricao",
        "dominio_regra",
        "origem_transformacao"
    )
    .orderBy("tabela", "ordem")
)

tabela,ordem,coluna,tipo_dado,permite_nulo,descricao,dominio_regra,origem_transformacao
silver_aerodromos,53,latitude_double,double,true,Latitude convertida para representação decimal.,Valor double ou nulo quando a conversão não é possível.,Conversão de bronze_aerodromos.latitude.
silver_aerodromos,54,longitude_double,double,true,Longitude convertida para representação decimal.,Valor double ou nulo quando a conversão não é possível.,Conversão de bronze_aerodromos.longitude.
silver_aerodromos,55,altitude_double,double,true,Altitude convertida para representação numérica.,Valor double ou nulo quando a conversão não é possível.,Conversão de bronze_aerodromos.altitude.
silver_vra,24,partida_prevista_ts,timestamp,true,Data e hora previstas para a partida convertidas para timestamp.,Timestamp ou nulo quando o valor original não está disponível.,Conversão de bronze_vra.partida_prevista.
silver_vra,25,partida_real_ts,timestamp,true,Data e hora efetivamente registradas para a partida convertidas para timestamp.,Timestamp ou nulo.,Conversão de bronze_vra.partida_real.
silver_vra,26,chegada_prevista_ts,timestamp,true,Data e hora previstas para a chegada convertidas para timestamp.,Timestamp ou nulo.,Conversão de bronze_vra.chegada_prevista.
silver_vra,27,chegada_real_ts,timestamp,true,Data e hora efetivamente registradas para a chegada convertidas para timestamp.,Timestamp ou nulo.,Conversão de bronze_vra.chegada_real.
silver_vra,28,referencia_ts,timestamp,true,Data de referência do registro convertida para timestamp.,Timestamp derivado do campo referencia.,Conversão de bronze_vra.referencia.
silver_vra,29,numero_de_assentos_int,int,true,Número de assentos convertido para tipo inteiro.,Inteiro ou nulo quando a conversão não é possível.,Conversão de bronze_vra.numero_de_assentos.
silver_vra,30,atraso_partida_min,double,true,Diferença em minutos entre partida real e partida prevista.,"Positivo para atraso, negativo para antecipação e nulo quando não calculável.",Diferença entre partida_real_ts e partida_prevista_ts.


In [0]:
comentarios_tabelas_silver = {
    "silver_vra": "Camada tratada do VRA, com deduplicação, tipagem e atributos derivados para análise.",
    "silver_aerodromos": "Camada tratada do cadastro de aeródromos, com deduplicação e conversão de coordenadas e altitude."
}

for tabela, comentario in comentarios_tabelas_silver.items():
    comentario_sql = comentario.replace("'", "''")

    spark.sql(
        f"""
        COMMENT ON TABLE workspace.mvp_sprint_3_anac.{tabela}
        IS '{comentario_sql}'
        """
    )

for registro in catalogo_semantico_silver:
    tabela = registro["tabela"]
    coluna = registro["coluna"]
    descricao = registro["descricao"].replace("'", "''")

    spark.sql(
        f"""
        ALTER TABLE workspace.mvp_sprint_3_anac.{tabela}
        ALTER COLUMN `{coluna}`
        COMMENT '{descricao}'
        """
    )

print("Comentários das tabelas e campos derivados da Silver registrados com sucesso no Unity Catalog.")

Comentários das tabelas e campos derivados da Silver registrados com sucesso no Unity Catalog.


In [0]:
comentarios_tabelas_bronze = {
    "bronze_vra": "Camada bruta dos dados de Voos Regulares Ativos da ANAC, preservada próxima ao formato original e acrescida de metadados técnicos de ingestão.",
    "bronze_aerodromos": "Camada bruta do cadastro de aeródromos públicos da ANAC, preservada próxima ao formato original e acrescida de metadados técnicos de ingestão."
}

for tabela, comentario in comentarios_tabelas_bronze.items():
    comentario_sql = comentario.replace("'", "''")

    spark.sql(
        f"""
        COMMENT ON TABLE workspace.mvp_sprint_3_anac.{tabela}
        IS '{comentario_sql}'
        """
    )

print("Comentários das tabelas Bronze registrados com sucesso no Unity Catalog.")

Comentários das tabelas Bronze registrados com sucesso no Unity Catalog.


In [0]:
catalogo_semantico_bronze = [
    Row(
        tabela="bronze_vra",
        coluna="arquivo_origem",
        descricao="Nome do arquivo CSV de origem do registro.",
        dominio_regra="Nome do arquivo mensal VRA utilizado na ingestão.",
        origem_transformacao="Metadado técnico obtido a partir de _metadata.file_name durante a ingestão."
    ),
    Row(
        tabela="bronze_vra",
        coluna="data_ingestao",
        descricao="Data e hora em que o registro foi ingerido para a camada Bronze.",
        dominio_regra="Timestamp gerado no momento da ingestão.",
        origem_transformacao="Gerado com current_timestamp() durante a ingestão."
    ),
    Row(
        tabela="bronze_aerodromos",
        coluna="arquivo_origem",
        descricao="Nome do arquivo CSV de origem do registro.",
        dominio_regra="Nome do arquivo de aeródromos utilizado na ingestão.",
        origem_transformacao="Metadado técnico obtido a partir de _metadata.file_name durante a ingestão."
    ),
    Row(
        tabela="bronze_aerodromos",
        coluna="data_ingestao",
        descricao="Data e hora em que o registro foi ingerido para a camada Bronze.",
        dominio_regra="Timestamp gerado no momento da ingestão.",
        origem_transformacao="Gerado com current_timestamp() durante a ingestão."
    )
]

df_catalogo_semantico_bronze = spark.createDataFrame(catalogo_semantico_bronze)

display(
    df_catalogo_semantico_bronze
    .join(
        df_catalogo_campos.select(
            "tabela", "ordem", "coluna", "tipo_dado", "permite_nulo"
        ),
        on=["tabela", "coluna"],
        how="left"
    )
    .select(
        "tabela",
        "ordem",
        "coluna",
        "tipo_dado",
        "permite_nulo",
        "descricao",
        "dominio_regra",
        "origem_transformacao"
    )
    .orderBy("tabela", "ordem")
)

tabela,ordem,coluna,tipo_dado,permite_nulo,descricao,dominio_regra,origem_transformacao
bronze_aerodromos,51,arquivo_origem,string,true,Nome do arquivo CSV de origem do registro.,Nome do arquivo de aeródromos utilizado na ingestão.,Metadado técnico obtido a partir de _metadata.file_name durante a ingestão.
bronze_aerodromos,52,data_ingestao,timestamp,true,Data e hora em que o registro foi ingerido para a camada Bronze.,Timestamp gerado no momento da ingestão.,Gerado com current_timestamp() durante a ingestão.
bronze_vra,22,arquivo_origem,string,true,Nome do arquivo CSV de origem do registro.,Nome do arquivo mensal VRA utilizado na ingestão.,Metadado técnico obtido a partir de _metadata.file_name durante a ingestão.
bronze_vra,23,data_ingestao,timestamp,true,Data e hora em que o registro foi ingerido para a camada Bronze.,Timestamp gerado no momento da ingestão.,Gerado com current_timestamp() durante a ingestão.


In [0]:
for registro in catalogo_semantico_bronze:
    tabela = registro["tabela"]
    coluna = registro["coluna"]
    descricao = registro["descricao"].replace("'", "''")

    spark.sql(
        f"""
        ALTER TABLE workspace.mvp_sprint_3_anac.{tabela}
        ALTER COLUMN `{coluna}`
        COMMENT '{descricao}'
        """
    )

print("Comentários dos campos técnicos da Bronze registrados com sucesso no Unity Catalog.")

Comentários dos campos técnicos da Bronze registrados com sucesso no Unity Catalog.
